<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Chain mit LangChain
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M13-RAG-Chain"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

In [ ]:
#@title 📂 Dokumente, Bilder { display-mode: "form" }
!rm -rf /content/files
!mkdir /content/files
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_1.txt  -o /content/files/biografien_1.txt
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_2.md   -o /content/files/biografien_2.md
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_3.pdf  -o /content/files/biografien_3.pdf
!curl -L https://raw.githubusercontent.com/ralf-42/Agenten/main/02_daten/01_text/biografien_4.docx -o /content/files/biografien_4.docx

# 1 | Übersicht
---



In *RAG-Konzepte & Embeddings* haben wir Embeddings verstanden, in *ChromaDB & Indexing* ChromaDB kennengelernt.
Jetzt **verbinden** wir alles: eine vollständige **RAG-Chain** mit LangChain.

**RAG-Chain vs. RAG-Agent**

| Eigenschaft | RAG-Chain | RAG-Agent |
|-------------|-----------|----------|
| Ablauf | Deterministisch (immer gleich) | Dynamisch (entscheidet selbst) |
| Tool-Nutzung | Nein | Ja, inklusive RAG als Tool |
| Kontrolle | Vollständig | Flexibel, aber weniger vorhersehbar |
| Einstieg | ✅ Einfacher | Komplexer (M11) |

**Was wir bauen**

Eine RAG-Chain, die:
1. Dokumente in ChromaDB indexiert (Phase 1)
2. Bei einer Anfrage relevante Chunks abruft (Retrieval)
3. Den LLM mit dem Kontext aufruft und antwortet (Generation)

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart LR
    subgraph Setup["Setup (einmalig)"]
        DOCS["📄 Dokumente"] --> SPLIT["✂️ Splitter"]
        SPLIT --> EMBED["🔢 Embeddings"]
        EMBED --> DB[("🗄️ ChromaDB")]
    end

    subgraph Chain["RAG-Chain (LCEL)"]
        Q["❓ Frage"] --> RET["🔍 Retriever"]
        DB --> RET
        RET --> FMT["📝 format_docs()"]
        Q --> PROMPT["📋 RAG-Prompt"]
        FMT --> PROMPT
        PROMPT --> LLM["🤖 LLM"]
        LLM --> PARSE["✂️ StrOutputParser"]
        PARSE --> ANS["💬 Antwort"]
    end
'''

mermaid(diagram, width=1100)

# 2 | Retriever erstellen
---



Ein **Retriever** ist die Schnittstelle zwischen der Vektordatenbank und der Chain.  
Er empfängt eine Frage und gibt die relevantesten Dokumente zurück.

<p><font color='black' size="5">
Wissensbasis aufbauen
</font></p>



Wir nutzen Agenten-Kurs-Texte als Wissensbasis:

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# LLM und Embeddings initialisieren
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*triton.*")

from langchain_community.document_loaders import (
    UnstructuredMarkdownLoader,
    UnstructuredWordDocumentLoader,
    PyPDFLoader,
    UnstructuredFileLoader,
    DirectoryLoader,
)

# Loader-Konfiguration
loader_mapping = {
    "*.md":   UnstructuredMarkdownLoader,
    "*.docx": UnstructuredWordDocumentLoader,
    "*.pdf":  PyPDFLoader,
    "*.txt":  UnstructuredFileLoader,
}

def load_documents_from_directory(directory_path):
    """Lädt Dokumente aus dem Verzeichnis für alle unterstützten Dateitypen."""
    documents = []
    for file_pattern, loader_cls in loader_mapping.items():
        loader = DirectoryLoader(directory_path, glob=file_pattern, loader_cls=loader_cls)
        documents.extend(loader.load())
    return documents

# Dokumente laden + splitten
dokumente_raw = load_documents_from_directory("/content/files")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(dokumente_raw)

# ChromaDB aufbauen (persistent)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="kurs_wissensbasis",
    persist_directory="chroma_m13"
)

print(f"Geladene Dokumente: {len(dokumente_raw)}")
print(f"Chunks indexiert:   {vectorstore._collection.count()}")

# 3 | Similarity Search
---



Bevor wir die vollständige RAG-Chain bauen, testen wir den Retriever direkt.

<p><font color='black' size="5">
Retriever-Konfiguration
</font></p>

| Parameter | Beschreibung | Wert |
|-----------|-------------|------|
| `k` | Anzahl zurückgegebener Dokumente | 3–5 |
| `search_type` | Suchalgorithmus | `"similarity"` (Standard) |
| `score_threshold` | Mindest-Ähnlichkeit (0–1) | Optional, z.B. 0.5 |

In [ ]:
# Retriever aus dem Vectorstore erstellen
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Testanfrage
test_frage = "Welche Methode hat Nadia Khoury entwickelt, um Mondstaub zu nutzen, und für welches Projekt arbeitet sie?"
gefundene_docs = retriever.invoke(test_frage)

print(f"Frage: {test_frage}\n")
print(f"Gefundene Dokumente: {len(gefundene_docs)}\n")
for i, doc in enumerate(gefundene_docs):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"Dokument {i+1}:")
    print(f"  Quelle:  {quelle}")
    print(f"  Inhalt:  {doc.page_content[:150]}...")
    print()

In [ ]:
# Similarity Search mit Score (fuer Debugging)
ergebnisse = vectorstore.similarity_search_with_score(
    "Was hat Dorian Blackwood vor seiner Karriere bei IllusionSec gemacht, und wie hilft er heute Kindern?",
    k=3
)

mprint("## Similarity Search mit Scores\n")
rows = ["| Nr | Quelle | Score | Inhalt |", "|-----|--------|-------|--------|"]
for i, (doc, score) in enumerate(ergebnisse):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    inhalt = doc.page_content[:60].replace("\n", " ")
    rows.append(f"| {i+1} | {quelle} | {score:.4f} | {inhalt}... |")
mprint("\n".join(rows))

# 4 | RAG-Chain bauen (LCEL)
---



Aufbau einer vollständigen RAG-Chain mit **LCEL** (LangChain Expression Language).



<p><font color='black' size="5">
Chain-Architektur
</font></p>


```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
```

**Erklärung:**
1. `retriever | format_docs` – Holt Dokumente und formatiert sie als Text
2. `RunnablePassthrough()` – Leitet die Frage unverändert weiter
3. `rag_prompt` – Baut den Prompt mit Kontext und Frage
4. `llm` – Sendet an GPT-4o-mini
5. `StrOutputParser()` – Extrahiert den Text aus der Antwort

In [ ]:
# RAG-Prompt aus Prompt-Template laden
rag_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m13_rag_prompt.md", mode="T")

# Hilfsfunktion: Dokumente formatieren
def format_docs(docs):
    return (chr(10) * 2).join(doc.page_content for doc in docs)

# RAG-Chain zusammenbauen (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG-Chain bereit")

In [ ]:
# RAG-Chain testen
test_fragen = [
    "Welche Methode hat Nadia Khoury entwickelt, um Mondstaub zu nutzen, und für welches Projekt arbeitet sie?",
    "Was hat Dorian Blackwood vor seiner Karriere bei IllusionSec gemacht, und wie hilft er heute Kindern?",
    "Welche zwei Fachrichtungen verbindet Raoul Mendez in seiner Arbeit, und was hat sein Projekt EarthEars bereits entdeckt?",
]

for frage in test_fragen:
    print(f"Frage: {frage}")
    antwort = rag_chain.invoke(frage)
    print(f"Antwort: {antwort}")
    print("-" * 60)
    print()

# 5 | RAG im LangSmith
---



LangSmith zeigt den vollständigen Trace der RAG-Chain – ideal zum Debuggen, wenn Antworten nicht passen.

**Was LangSmith anzeigt**

| Ebene | Inhalt | Nützlich für |
|-------|--------|-------------|
| **Chain-Ebene** | Input/Output der gesamten Chain | Gesamtbild |
| **Retriever-Ebene** | Welche Dokumente gefunden wurden | Retrieval-Qualität |
| **Prompt-Ebene** | Exakter Prompt (mit Kontext) | Prompt-Debugging |
| **LLM-Ebene** | Tokens, Kosten, Latenz | Performance |

**Trace mit run_name**

```python
antwort = rag_chain.with_config(run_name="M13-Kap5-RAG-Chain").invoke(frage)
```

In [ ]:
# RAG mit LangSmith-Tracing und run_name
run_cfg = {"run_name": "M13-Kap5-RAG-Chain", "tags": ["rag", "m13"]}

frage = "Welche zwei Fachrichtungen verbindet Raoul Mendez in seiner Arbeit, und was hat sein Projekt EarthEars bereits entdeckt?"

antwort = rag_chain.with_config(**run_cfg).invoke(frage)

# Gefundene Dokumente separat anzeigen
kontext_docs = retriever.invoke(frage)

mprint(f"""
## RAG-Ergebnis mit LangSmith-Trace

**Frage:** {frage}

**Verwendete Quellen ({len(kontext_docs)} Dokumente):**
""")

for i, doc in enumerate(kontext_docs):
    quelle = doc.metadata.get("source", "?").split("/")[-1]
    print(f"  {i+1}. {quelle} – {doc.page_content[:80]}...")

mprint(f"""

**Antwort:**

{antwort}

> Trace verfuegbar in LangSmith unter Projekt: M13-RAG-Chain
""")

# 6 | RAG-Optimierung
---



*Häufige Probleme und Lösungen*

| Problem | Symptom | Lösung |
|---------|---------|--------|
| Falsche Dokumente | Irrelevante Antworten | Chunk-Größe reduzieren, k erhöhen |
| Halluzinationen | Fakten nicht in Quellen | Temperatur auf 0.0, strengere Prompts |
| Langsame Antworten | Hohe Latenz | Weniger Chunks (k), kleinere Modelle |
| Zu viel Kontext | Token-Limit | Komprimierung, k reduzieren |
| Inkonsistente Antworten | Unterschiedliche Resultate | temperature=0.0 setzen |

**Optimierungsstrategie**

```
1. Mit kleinem k starten (k=3)
2. LangSmith-Traces analysieren
3. Retrieval-Qualität prüfen (werden relevante Docs gefunden?)
4. Prompt anpassen (präziser, Quellenbindung erzwingen)
5. chunk_size experimentieren (200–500)
```

In [ ]:
# Optimierter Retriever mit Score-Threshold
retriever_opt = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

# Optimierte RAG-Chain
rag_chain_opt = (
    {"context": retriever_opt | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Vergleich: Standard vs. Optimiert
frage_test = "Erklaere mir, was ein Extremwetter-Fotografin ist. Kurzer Text reicht."

docs_std = retriever.invoke(frage_test)
docs_opt = retriever_opt.invoke(frage_test)

print(f"Standard Retriever (k=3):    {len(docs_std)} Dokumente")
print(f"Optimierter Retriever (k=4): {len(docs_opt)} Dokumente")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M13-RAG-Chain", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**
- Laden Sie die Biografien-Dateien aus `../02_daten/01_text/` (z. B. `biografien_1.txt`, `biografien_2.md`).
- Erstellen Sie eine persistente ChromaDB-Collection `biografien`.
- Testen Sie den Retriever mit `retriever.invoke("Was ist RAG?")` und geben Sie die Treffer aus.

**Aufbau**
- Bauen Sie eine RAG-Chain mit dem Standard-RAG-Prompt (`load_prompt("../05_prompt/m10_rag_prompt.md")`).
- Beantworten Sie 3 Fragen zu den Biografien und zeigen Sie die Quellen an.
- Speichern Sie Retriever in `retriever` und die Chain in `rag_chain`.

**Vertiefung**
- Fuegen Sie LangSmith-Tracing mit `run_name` und `tags` hinzu.
- Analysieren Sie in LangSmith, welche Dokumente bei welcher Frage gefunden wurden.
- Begruenden Sie in 2 Saetzen, welche Dokument-Chunks am haufigsten retrieviert werden und warum.

In [ ]:
# ✅ Selbstcheck Grundlagen 🟢
# Führen Sie diesen Block aus, nachdem Sie Retriever und Collection aufgebaut haben.
# Passen Sie die Variablennamen ggf. an Ihre Implementierung an.

assert retriever is not None, "retriever ist None – bitte aus dem Vectorstore erstellen."
docs = retriever.invoke("Was ist RAG?")
assert len(docs) > 0, "retriever.invoke() liefert keine Dokumente – Collection befüllt?"
print("✅ Grundlagen-Selbstcheck bestanden!")

In [ ]:
# ✅ Selbstcheck Aufbau 🟡
# Führen Sie diesen Block aus, nachdem Sie die RAG-Chain gebaut und eine Frage beantwortet haben.
# 'answer' muss die Antwort der RAG-Chain als String enthalten.

assert rag_chain is not None, "rag_chain ist None – bitte die LCEL-Chain zusammenbauen."
answer = rag_chain.invoke("Was sind die Hauptthemen der Biografie-Texte?")
assert isinstance(answer, str) and len(answer) > 10, "answer ist zu kurz oder kein String – RAG-Chain prüfen."
print("✅ Aufbau-Selbstcheck bestanden!")

In [ ]:
# ✅ Selbstcheck Vertiefung 🔴
import os

assert os.environ.get("LANGSMITH_TRACING") == "true", "LANGSMITH_TRACING ist nicht aktiv – Umgebungsvariable prüfen."
print("✅ Vertiefung-Selbstcheck bestanden!")